# 54. Celery Background Jobs — `celery`
**Python Libraries Lab · Phase 3 · Utilities & Testing**

_Run slow work in the background with a task queue._

⚙️ **Runs in Colab** after the setup cell below.

Some work is too slow to do while a customer waits, like sending emails, making invoices, or resizing images. Celery lets your app put that work on a queue and return immediately. Separate worker processes pick jobs off the queue and run them in the background, and they can retry jobs that fail.

**In this notebook:** You will define ShopEasy tasks for invoices and emails, start a real Celery worker in the background of Colab, send jobs and collect results, watch a flaky task retry automatically, and combine tasks into chains and groups.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
Celery passes jobs through a broker. The next cell installs and starts Redis inside Colab for that.

The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q celery redis
import importlib.metadata as md
for pkg in ['celery', 'redis']:
    print(pkg, md.version(pkg))

In [ ]:
# Celery needs a message broker. We install and start Redis inside Colab (about 20 seconds).
!apt-get -qq install -y redis-server > /dev/null 2>&1
!redis-server --daemonize yes --port 6379
import time; time.sleep(1)
!redis-cli ping

## Step 1 · Define tasks
Saved as `tasks.py` so the worker can import it. Redis is both broker and result store.

In [ ]:
%%writefile tasks.py
import time, random
from celery import Celery

app = Celery("shop", broker="redis://localhost:6379/0", backend="redis://localhost:6379/1")
app.conf.update(task_track_started=True, result_expires=3600)

@app.task
def make_invoice(order_id, amount):
    time.sleep(1)                      # pretend to build a PDF
    return {"order_id": order_id, "invoice": f"INV-{order_id}", "amount": amount}

@app.task
def send_email(invoice):
    time.sleep(0.5)
    return f"emailed {invoice['invoice']} for Rs.{invoice['amount']}"

@app.task(bind=True, max_retries=5, default_retry_delay=0.5)
def charge_card(self, order_id):
    random.seed(order_id + self.request.retries)
    if random.random() < 0.6:          # the payment gateway is flaky
        raise self.retry(exc=ConnectionError("gateway timeout"))
    return f"charged order {order_id} after {self.request.retries} retries"

@app.task
def add(a, b):
    return a + b

## Step 2 · Start a worker
A worker is a separate process that waits for jobs. We start one in the background.

In [ ]:
import subprocess, time
worker = subprocess.Popen(["celery", "-A", "tasks", "worker", "--loglevel=INFO", "--concurrency=4", "--pool=threads"],
                          stdout=open("worker.log", "w"), stderr=subprocess.STDOUT)
time.sleep(6)
print([l for l in open("worker.log").read().splitlines() if "ready" in l or "[tasks]" in l][-1:])

## Step 3 · Send jobs and get results
`.delay()` returns immediately. `.get()` waits for the result.

In [ ]:
import importlib, tasks
importlib.reload(tasks)
t = time.perf_counter()
jobs = [tasks.make_invoice.delay(1000 + i, 500 * (i + 1)) for i in range(4)]
print(f"4 jobs queued in {time.perf_counter() - t:.3f}s; status: {jobs[0].status}")
results = [j.get(timeout=20) for j in jobs]
print(f"All done after {time.perf_counter() - t:.1f}s (4 workers ran them in parallel)")
for r in results:
    print(r)

## Step 4 · Automatic retries
The flaky payment task fails sometimes and Celery retries it for us.

In [ ]:
for order in [2001, 2002, 2003]:
    print(tasks.charge_card.delay(order).get(timeout=30))

## Step 5 · Chains and groups
A chain passes each result to the next task. A group runs tasks in parallel.

In [ ]:
from celery import chain, group
flow = chain(tasks.make_invoice.s(3001, 2499), tasks.send_email.s())
print(flow.apply_async().get(timeout=20))
print(group(tasks.add.s(i, i) for i in range(6)).apply_async().get(timeout=20))

## Step 6 · Stop the worker
Clean up the background process.

In [ ]:
worker.terminate()
print("Worker stopped")

## ✍️ Your turn
Start a new worker if needed, then call `tasks.add.delay(20, 22)` and store the **result** in `answer`. (The cell restarts the worker for you.)

Write your code in the next cell, run it, then run the check cell.

In [ ]:
worker = subprocess.Popen(["celery", "-A", "tasks", "worker", "--pool=threads", "--concurrency=2"],
                          stdout=open("worker2.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
answer = None  # your code here

In [ ]:
def check():
    return answer == 42, f"Expected 42, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = tasks.add.delay(20, 22).get(timeout=20)
worker.terminate()

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Task**
- **Broker**
- **Worker**
- **.delay()**
- **Result backend**
- **Retry**
- **Chain / group**

**Next:** 55. Excel Files (`openpyxl · xlsxwriter`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/55_openpyxl_xlsxwriter.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install celery redis
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.